# Multilabel Emotion Classification — Gradient Boosting
Dataset: `Jsevisal/go_emotions_wheel` | Model: `OneVsRestClassifier(GradientBoostingClassifier)`

In [ ]:
import numpy as np
import polars as pl
import pandas as pd
from datasets import load_dataset

In [ ]:
ds = load_dataset("Jsevisal/go_emotions_wheel")
df_train      = pl.from_dicts(ds['train'][:])
df_test       = pl.from_dicts(ds['test'][:])
df_validation = pl.from_dicts(ds['validation'][:])

In [ ]:
EXCLUDED = [1, 5, 6]

def filter_labels(df, excluded):
    mask = ~pl.lit(False)
    for lbl in excluded:
        mask = mask & ~pl.col("labels").list.contains(lbl)
    return df.filter(mask)

df_train      = filter_labels(df_train,      EXCLUDED)
df_test       = filter_labels(df_test,       EXCLUDED)
df_validation = filter_labels(df_validation, EXCLUDED)

In [ ]:
label_dict = {
    0: 'joy',
    2: 'fear',
    3: 'surprise',
    4: 'sadness',
    7: 'anger',
    8: 'disgust'
}

In [ ]:
from sentence_transformers import SentenceTransformer

model_st = SentenceTransformer('all-MiniLM-L6-v2')

X_train = model_st.encode(df_train['text'].to_list(),      batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_test  = model_st.encode(df_test['text'].to_list(),       batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_val   = model_st.encode(df_validation['text'].to_list(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

y_train_raw = df_train['labels'].to_list()
y_test_raw  = df_test['labels'].to_list()
y_val_raw   = df_validation['labels'].to_list()

all_labels_combined  = y_train_raw + y_test_raw + y_val_raw
all_unique_label_ids = sorted(set(item for sublist in all_labels_combined for item in sublist))

mlb = MultiLabelBinarizer(classes=all_unique_label_ids)
mlb.fit(all_labels_combined)

y_train = mlb.transform(y_train_raw)
y_test  = mlb.transform(y_test_raw)
y_val   = mlb.transform(y_val_raw)

print("Label order:", [label_dict[c] for c in mlb.classes_])
print("y_train shape:", y_train.shape)

In [ ]:
label_counts = pd.DataFrame(
    y_train.sum(axis=0),
    index=[label_dict[c] for c in mlb.classes_],
    columns=['count']
)
print(label_counts.sort_values('count', ascending=False))

## 1. Baseline Gradient Boosting

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import (
    hamming_loss, jaccard_score, classification_report,
    accuracy_score, f1_score
)

gb_base = OneVsRestClassifier(
    GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )
)

gb_base.fit(X_train, y_train)
y_pred_base = gb_base.predict(X_test)

print("=== Baseline Gradient Boosting ===")
print(f"Hamming Loss:       {hamming_loss(y_test, y_pred_base):.4f}  (↓ lower is better)")
print(f"Jaccard (macro):    {jaccard_score(y_test, y_pred_base, average='macro'):.4f}  (↑ higher is better)")
print(f"Jaccard (weighted): {jaccard_score(y_test, y_pred_base, average='weighted'):.4f}")
print(f"Subset Accuracy:    {accuracy_score(y_test, y_pred_base):.4f}")
print(f"F1 (macro):         {f1_score(y_test, y_pred_base, average='macro', zero_division=0):.4f}")
print()
print(classification_report(
    y_test, y_pred_base,
    target_names=[label_dict[c] for c in mlb.classes_],
    zero_division=0
))

## 2. Hyperparameter Tuning with Optuna

In [ ]:
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    n_estimators   = trial.suggest_int('n_estimators', 50, 300)
    learning_rate  = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth      = trial.suggest_int('max_depth', 2, 8)
    subsample      = trial.suggest_float('subsample', 0.5, 1.0)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features   = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    clf = OneVsRestClassifier(
        GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            subsample=subsample,
            min_samples_leaf=min_samples_leaf,
            max_features=max_features,
            random_state=42
        )
    )
    clf.fit(X_train, y_train)
    y_pred_val = clf.predict(X_val)
    return f1_score(y_val, y_pred_val, average='macro', zero_division=0)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Best params:", study.best_params)
print(f"Best val F1 (macro): {study.best_value:.4f}")

In [ ]:
best = study.best_params

gb_best = OneVsRestClassifier(
    GradientBoostingClassifier(
        n_estimators=best['n_estimators'],
        learning_rate=best['learning_rate'],
        max_depth=best['max_depth'],
        subsample=best['subsample'],
        min_samples_leaf=best['min_samples_leaf'],
        max_features=best['max_features'],
        random_state=42
    )
)

gb_best.fit(X_train, y_train)
y_pred_best = gb_best.predict(X_test)

print("=== Tuned Gradient Boosting ===")
print(f"Hamming Loss:       {hamming_loss(y_test, y_pred_best):.4f}")
print(f"Jaccard (macro):    {jaccard_score(y_test, y_pred_best, average='macro'):.4f}")
print(f"Jaccard (weighted): {jaccard_score(y_test, y_pred_best, average='weighted'):.4f}")
print(f"Subset Accuracy:    {accuracy_score(y_test, y_pred_best):.4f}")
print(f"F1 (macro):         {f1_score(y_test, y_pred_best, average='macro', zero_division=0):.4f}")
print()
print(classification_report(
    y_test, y_pred_best,
    target_names=[label_dict[c] for c in mlb.classes_],
    zero_division=0
))

## 3. Per-Label Threshold Tuning (on Validation set)

In [ ]:
def find_best_thresholds(y_true, y_proba, num_labels):
    best_thresholds = []
    for i in range(num_labels):
        thresholds = np.arange(0.10, 0.90, 0.05)
        best_score, best_th = 0, 0.5
        for th in thresholds:
            y_pred_i = (y_proba[:, i] >= th).astype(int)
            score = f1_score(y_true[:, i], y_pred_i, zero_division=0)
            if score > best_score:
                best_score, best_th = score, th
        best_thresholds.append(best_th)
    return best_thresholds

y_proba_val    = gb_best.predict_proba(X_val)
opt_thresholds = find_best_thresholds(y_val, y_proba_val, y_val.shape[1])

print("--- Optimal thresholds per label ---")
for i, label in enumerate([label_dict[c] for c in mlb.classes_]):
    print(f"  {label}: {opt_thresholds[i]:.2f}")

In [ ]:
y_proba_test = gb_best.predict_proba(X_test)
y_pred_tuned = np.zeros_like(y_proba_test)
for i, th in enumerate(opt_thresholds):
    y_pred_tuned[:, i] = (y_proba_test[:, i] >= th).astype(int)

print("=== Tuned GB + Per-label Thresholds ===")
print(f"Hamming Loss:       {hamming_loss(y_test, y_pred_tuned):.4f}")
print(f"Jaccard (macro):    {jaccard_score(y_test, y_pred_tuned, average='macro'):.4f}")
print(f"Jaccard (weighted): {jaccard_score(y_test, y_pred_tuned, average='weighted'):.4f}")
print(f"Subset Accuracy:    {accuracy_score(y_test, y_pred_tuned):.4f}")
print(f"F1 (macro):         {f1_score(y_test, y_pred_tuned, average='macro', zero_division=0):.4f}")
print()
print(classification_report(
    y_test, y_pred_tuned,
    target_names=[label_dict[c] for c in mlb.classes_],
    zero_division=0
))

## 4. ROC Curves per Emotion

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

n_classes = y_test.shape[1]
colors    = ['blue', 'red', 'green', 'purple', 'orange', 'brown']

plt.figure(figsize=(10, 8))
for i, color in zip(range(n_classes), colors):
    name = label_dict[mlb.classes_[i]]
    fpr_i, tpr_i, roc_ths = roc_curve(y_test[:, i], y_proba_test[:, i])
    roc_auc_i = auc(fpr_i, tpr_i)
    plt.plot(fpr_i, tpr_i, color=color, lw=2, label=f'{name} (AUC={roc_auc_i:.2f})')
    idx = np.argmin(np.abs(roc_ths - opt_thresholds[i]))
    plt.plot(fpr_i[idx], tpr_i[idx], 'o', color=color, markersize=9, markeredgecolor='black')

plt.plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
plt.xlim([-0.02, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — Gradient Boosting (tuned thresholds marked)', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('roc_curve_gradient_boosting.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: roc_curve_gradient_boosting.png")

## 5. Model Comparison Summary

In [ ]:
configs = [
    ("GB Baseline",           y_pred_base),
    ("GB Tuned (Optuna)",     y_pred_best),
    ("GB Tuned + Thresholds", y_pred_tuned),
]

rows = []
for name, y_pred in configs:
    rows.append({
        "Model":              name,
        "Hamming Loss":       round(hamming_loss(y_test, y_pred), 4),
        "Jaccard (macro)":    round(jaccard_score(y_test, y_pred, average='macro'), 4),
        "Jaccard (weighted)": round(jaccard_score(y_test, y_pred, average='weighted'), 4),
        "Subset Accuracy":    round(accuracy_score(y_test, y_pred), 4),
        "F1 (macro)":         round(f1_score(y_test, y_pred, average='macro', zero_division=0), 4),
    })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))